In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [8]:
train_dataset = datasets.ImageFolder(
    root="../data/EuroSAT/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root="../data/EuroSAT/val",
    transform=val_transform
)

In [9]:
print(len(train_dataset))
print(len(val_dataset))
print(train_dataset.classes)

18900
4050
['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [10]:
print(os.path.exists("../data/EuroSAT/train"))
print(os.path.exists("../data/EuroSAT/val"))

True
True


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

In [12]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [13]:
os.listdir("../data/EuroSAT")

['val', 'train']

In [14]:
model = models.resnet18(weights="DEFAULT")

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/kartik/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:15<00:00, 2.94MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [15]:
model.fc = nn.Linear(
    model.fc.in_features,
    10
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [17]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print(outputs.shape)

torch.Size([32, 10])


In [18]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [19]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [20]:
epochs = 5

best_val_acc = 0

In [23]:
for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}/{epochs}"
    )

    print(
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "../models/rgb_best.pth"
        )

        print("Best model saved")

Epoch 1/5
Train Loss: 0.0492 | Train Acc: 98.41%
Val Loss: 0.0905 | Val Acc: 97.06%
Epoch 2/5
Train Loss: 0.0516 | Train Acc: 98.30%
Val Loss: 0.0731 | Val Acc: 97.36%
Best model saved
Epoch 3/5
Train Loss: 0.0341 | Train Acc: 98.89%
Val Loss: 0.0945 | Val Acc: 97.16%
Epoch 4/5
Train Loss: 0.0375 | Train Acc: 98.72%
Val Loss: 0.0936 | Val Acc: 96.99%
Epoch 5/5
Train Loss: 0.0338 | Train Acc: 98.92%
Val Loss: 0.0794 | Val Acc: 97.51%
Best model saved


In [25]:
import os

print(os.path.exists("../models/rgb_best.pth"))

True


In [26]:
os.path.exists("../models/rgb_best.pth")

True